# Arabic Toxicity Detection
Dataset: `textdetox/multilingual_toxicity_dataset` (Arabic split)

**Pipeline:** Setup → Load data → EDA → Cleaning → Split → Word2Vec → Embedding matrix → Padding

## 0. Setup
شغلي cell الـ install مرة وحدة، وبعدها **Runtime → Restart session**، وبعدها كملي من cell الـ imports.

In [ ]:
!pip install -q gensim

In [ ]:
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from tensorflow.keras.utils import pad_sequences, to_categorical

## 1. Load data

In [ ]:
ds = load_dataset("textdetox/multilingual_toxicity_dataset", split="ar")
df = ds.to_pandas()   # الأعمدة: text, toxic
df.head()

## 2. EDA

### 2.1 نظرة عامة: الحجم، القيم الناقصة، المكرر

In [ ]:
print("Shape:", df.shape)
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated(subset="text").sum())

In [ ]:
df.sample(10, random_state=42)

### 2.2 توزيع الـ classes
إذا الـ classes متوازنة، الـ accuracy تكون metric مناسب.

In [ ]:
print(df["toxic"].value_counts())

df["toxic"].value_counts().plot(kind="bar", color=["steelblue", "indianred"])
plt.title("Class distribution")
plt.xticks([0, 1], ["Non-toxic (0)", "Toxic (1)"], rotation=0)
plt.show()

### 2.3 طول النصوص
يساعدنا نحدد `MAX_LEN` بعدين.

In [ ]:
df["n_words"] = df["text"].str.split().str.len()

print(df.groupby("toxic")["n_words"].describe())

df[df["toxic"] == 0]["n_words"].clip(upper=100).hist(bins=50, alpha=0.6, label="Non-toxic")
df[df["toxic"] == 1]["n_words"].clip(upper=100).hist(bins=50, alpha=0.6, label="Toxic")
plt.xlabel("Words per text (clipped at 100)")
plt.legend()
plt.show()

## 3. Cleaning
نوحد شكل النص عشان الكلمة الوحدة ما تنكتب بأكثر من طريقة (مثلاً "أحمد" و "احمد").

In [ ]:
def clean_arabic(text):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+|\bURL\b", " ", text)   # الروابط
    text = re.sub(r"@\w+|\bUSER\b", " ", text)                # المنشن
    text = re.sub(r"[\u064B-\u0652\u0670]", "", text)        # التشكيل
    text = text.replace("ـ", "")                              # التطويل
    text = re.sub("[إأآ]", "ا", text)                         # توحيد الألف
    text = text.replace("ة", "ه")
    text = text.replace("ى", "ي").replace("ی", "ي")           # ي عربية وفارسية
    text = text.replace("ک", "ك")                             # ك فارسية
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)                # "هههههه" -> "هه"
    text = re.sub(r"[^\u0621-\u064A\s]", " ", text)           # نخلي الحروف العربية فقط (يشيل الترقيم والإيموجي و #)
    text = re.sub(r"\s+", " ", text).strip()                  # مسافات زيادة
    return text

df["clean"] = df["text"].apply(clean_arabic)

نحذف النصوص القصيرة جداً والمكرر **قبل** الـ split، عشان ما تطلع نفس الجملة في train و test (data leakage).

In [ ]:
df = df[df["clean"].str.split().str.len() >= 2]
df = df.drop_duplicates(subset="clean").reset_index(drop=True)

print("After cleaning:", df.shape)
print(df["toxic"].value_counts())

In [ ]:
# مقارنة قبل / بعد
for i in range(5):
    print("Before:", df["text"][i])
    print("After: ", df["clean"][i], "\n")

### 3.1 أكثر الكلمات تكراراً في كل class

In [ ]:
for label, name in [(1, "Toxic"), (0, "Non-toxic")]:
    texts = df[df["toxic"] == label]["clean"]
    words = Counter(w for t in texts for w in t.split())
    print(name, ":", [w for w, _ in words.most_common(15)], "\n")

**ملاحظة:** الكلمات الأكثر تكراراً مشتركة بين الـ classes (مثل: يا، من، و، في)، فهي ما تميز وحدها بين toxic و non-toxic.

## 4. Train / Validation / Test split
70% / 15% / 15% مع `stratify` عشان تبقى نسبة الـ classes نفسها في كل مجموعة.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["toxic"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["toxic"], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 5. Word2Vec
نتدرب على **train فقط**، لأن الموديل ما لازم يشوف أي شي من val أو test.

In [ ]:
train_sentences = [t.split() for t in train_df["clean"]]

w2v = Word2Vec(
    sentences=train_sentences,
    vector_size=100,   # طول المتجه لكل كلمة
    window=5,          # عدد الكلمات المحيطة اللي يتعلم منها
    min_count=2,       # نتجاهل الكلمات اللي ظهرت مرة وحدة
    sg=1,              # Skip-gram (أفضل للداتا الصغيرة)
    epochs=30,         # epochs أكثر لأن الداتا صغيرة
    seed=42,
)
print("Vocab size:", len(w2v.wv))

### 5.1 نختبر الـ embeddings: وش أقرب الكلمات؟

In [ ]:
for word in ["غبي", "كلب", "حقيره", "احبك", "صباح"]:
    if word in w2v.wv:
        similar = [w for w, _ in w2v.wv.most_similar(word, topn=6)]
        print(word, "→", similar)
    else:
        print(word, "→ غير موجودة")

**ملاحظة:** بعض النتائج منطقية (غبي → متخلف، صباح → الخير) وبعضها عشوائي. السبب إن الداتا صغيرة (~3,500 جملة)، فـ Word2Vec ما شاف أمثلة كافية لكل كلمة.

In [ ]:
w2v.save("word2vec_ar_toxic.model")

## 6. Vocabulary + Embedding matrix
- `word_index`: كل كلمة لها رقم. رقم 0 للـ padding، ورقم 1 للكلمات غير المعروفة (OOV).
- `embedding_matrix`: الصف رقم i = متجه الكلمة اللي رقمها i. هذي اللي بنحطها في الـ Embedding layer.

In [ ]:
word_index = {"<PAD>": 0, "<OOV>": 1}
for word in w2v.wv.index_to_key:
    word_index[word] = len(word_index)
index_word = {i: w for w, i in word_index.items()}

vocab_size = len(word_index)
EMB_DIM = w2v.vector_size
print("vocab_size:", vocab_size, "| EMB_DIM:", EMB_DIM)

In [ ]:
embedding_matrix = np.zeros((vocab_size, EMB_DIM))   # PAD و OOV يبقون أصفار
for word, i in word_index.items():
    if word in w2v.wv:
        embedding_matrix[i] = w2v.wv[word]

print("Shape:", embedding_matrix.shape)

# تأكد: متجه الكلمة في الـ matrix نفس اللي في Word2Vec
print("Match:", np.allclose(embedding_matrix[word_index["غبي"]], w2v.wv["غبي"]))

### 6.1 نسبة الكلمات غير المعروفة (OOV) في val و test

In [ ]:
def oov_rate(data):
    words = [w for t in data["clean"] for w in t.split()]
    unknown = [w for w in words if w not in word_index]
    return len(unknown) / len(words)

print(f"OOV rate — Val: {oov_rate(val_df):.1%} | Test: {oov_rate(test_df):.1%}")

**ملاحظة:** نسبة الـ OOV عالية (تقريباً ثلث الكلمات)، لأن الداتا صغيرة وفيها لهجات كثيرة (تونسي، لبناني، خليجي...). هذا قيد من الداتا نفسها، وبيأثر على أداء الموديل.

In [ ]:
np.save("embedding_matrix.npy", embedding_matrix)
with open("word_index.json", "w", encoding="utf-8") as f:
    json.dump(word_index, f, ensure_ascii=False)

## 7. Text → Sequences → Padding
الموديل يحتاج كل الجمل بنفس الطول. نختار `MAX_LEN` بحيث يغطي 95% من الجمل.

In [ ]:
train_lens = train_df["clean"].str.split().str.len()
MAX_LEN = int(np.percentile(train_lens, 95))
print("MAX_LEN:", MAX_LEN)
print(f"Sentences longer than MAX_LEN: {(train_lens > MAX_LEN).mean():.1%}")

In [ ]:
def to_padded(data):
    # كل كلمة → رقمها، والكلمة غير المعروفة → 1
    seqs = [[word_index.get(w, 1) for w in t.split()] for t in data["clean"]]
    return pad_sequences(seqs, maxlen=MAX_LEN, padding="post", truncating="post")

X_train = to_padded(train_df)
X_val   = to_padded(val_df)
X_test  = to_padded(test_df)

In [ ]:
y_train = to_categorical(train_df["toxic"], 2)
y_val   = to_categorical(val_df["toxic"], 2)
y_test  = to_categorical(test_df["toxic"], 2)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "| y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "| y_test: ", y_test.shape)

### 7.1 تأكد: نرجع الأرقام لكلمات

In [ ]:
i = 0
print("Text:   ", train_df["clean"].iloc[i])
print("Numbers:", X_train[i])
print("Back:   ", " ".join(index_word[t] for t in X_train[i] if t != 0))